In [ ]:
#@title Test model with lora
import torch
from transformers import AutoTokenizer
import auto_gptq
from auto_gptq import AutoGPTQForCausalLM, get_gptq_peft_model
from auto_gptq.utils.peft_utils import GPTQLoraConfig
#import gradio as gr

In [ ]:
model_name = 'fffrrt/ruGPT-3.5-13B-GPTQ'
model_basename = 'gptq_model-4bit-128g'

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoGPTQForCausalLM.from_quantized("gurgutan/ruGPT-13B-4bit",
        low_cpu_mem_usage=True,
        device_map='auto',
        trust_remote_code=True,
        inject_fused_attention = True,
        inject_fused_mlp = False,
        use_triton=True,
        warmup_triton=False,
        trainable=True)

In [ ]:
peft_config = GPTQLoraConfig(
    inference_mode=True,
)
model = get_gptq_peft_model(model, peft_config, 'models/Tinkoff_models_ruGPT-3.5/essays/checkpoint-200/adapter_model')

In [ ]:
model.is_loaded_in_4bit=True

In [ ]:
prompt = """
Напиши вопросы, над которыми стоит задуматься?
""".strip()

In [ ]:
encoded_input = tokenizer(prompt, return_tensors='pt').to('cuda:0')
output = model.generate(
    **encoded_input,
    max_new_tokens=100,
    do_sample=True,
    temperature=1,
)

In [ ]:
print(tokenizer.decode(output[0], skip_special_tokens=True))